In [12]:
# Run this only once if packages are missing:
# %pip install -q fasttext-wheel langdetect pandas
print('Using existing environment packages.')

Using existing environment packages.


## Language detection from lyrics

Two-pass detection strategy:

**Pass 1 — Script/character check (fast, rule-based)**
Scans the full lyrics for Unicode script ranges. If a dominant non-Latin
script is found above a character-count threshold, assign that language directly.
This handles multilingual songs that mix CJK/Japanese/Arabic characters into
otherwise Latin lyrics — FastText would misclassify these.

Priority order (first match wins):
- Japanese (hiragana/katakana present) → `ja`
- Chinese (CJK block, no kana) → `zh`
- Korean (Hangul) → `ko`
- Arabic → `ar`
- Thai → `th`
- Cyrillic → `ru`

**Pass 2 — FastText (for Latin-script and ambiguous lyrics)**
Songs that pass through the script check unchanged are fed to the FastText
lid model on the first 200 characters of lyrics.

Output: all columns from `lyrics.csv` + new `original_lang` column (ISO 639-1).


In [13]:
import urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import fasttext
import fasttext.FastText

# Idempotent monkey-patch: fix np.array(copy=False) error in NumPy 2.x
if not hasattr(fasttext.FastText._FastText, '_orig_predict'):
    fasttext.FastText._FastText._orig_predict = fasttext.FastText._FastText.predict

    def _safe_predict(self, text, k=1, threshold=0.0, on_unicode_error='strict'):
        import numpy as np
        _old_array = np.array
        def _array_compat(*args, **kwargs):
            kwargs.pop('copy', None)
            return _old_array(*args, **kwargs)
        np.array = _array_compat
        try:
            return fasttext.FastText._FastText._orig_predict(self, text, k=k, threshold=threshold, on_unicode_error=on_unicode_error)
        finally:
            np.array = _old_array

    fasttext.FastText._FastText.predict = _safe_predict
    print('NumPy 2.x patch applied.')
else:
    print('Patch already applied, skipping.')

PROJECT_ROOT = Path.cwd().parent  # go up from notebooks to project root

# -- Download FastText lid model if not cached --
MODEL_URL = 'https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz'
MODEL_PATH = PROJECT_ROOT / 'models' / 'lid.176.ftz'
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

if not MODEL_PATH.exists():
    print(f'Downloading FastText lid model -> {MODEL_PATH} ...')
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print('Done.')
else:
    print(f'Model already cached at {MODEL_PATH}')

ft_model = fasttext.load_model(str(MODEL_PATH))
print('FastText model loaded.')

Patch already applied, skipping.
Model already cached at d:\Users\Documents\GitHub\lyrics_analysis\models\lid.176.ftz
FastText model loaded.


In [ ]:
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
LYRICS_IN = PROCESSED_DIR / '01_lyrics.csv'
LANG_OUT  = PROCESSED_DIR / '02_lyrics_lang.csv'

# DATABRICKS PATH
# LYRICS_IN = '/Volumes/songs_db/default/storage/01_lyrics.csv'
# LANG_OUT  = '/Volumes/songs_db/default/storage/02_lyrics_lang.csv'

df = pd.read_csv(LYRICS_IN)
print(f'Loaded {len(df)} songs from {LYRICS_IN}')

# ── Pass 1: script/character check ───────────────────────────────────────
import unicodedata
import re

SCRIPT_THRESHOLD = 8  # min characters of a script to trigger assignment

def detect_by_script(lyrics: str) -> str | None:
    """
    Return ISO 639-1 code if a non-Latin script is dominant, else None.
    Priority: ja > zh > ko > ar > th > ru
    """
    if not isinstance(lyrics, str) or not lyrics.strip():
        return None
    s = lyrics

    # Unicode block regexes
    cnt_ja = len(re.findall(r'[\u3040-\u30ff\u31f0-\u31ff]', s))  # Hiragana/Katakana
    cnt_zh = len(re.findall(r'[\u4e00-\u9fff]', s))                # CJK ideographs
    cnt_ko = len(re.findall(r'[\uac00-\ud7af\u1100-\u11ff]', s))  # Hangul
    cnt_ar = len(re.findall(r'[\u0600-\u06ff\u0750-\u077f\u08a0-\u08ff]', s))
    cnt_th = len(re.findall(r'[\u0e00-\u0e7f]', s))
    cnt_ru = len(re.findall(r'[\u0400-\u04ff]', s))                # Cyrillic

    script_counts = {
        'ja': cnt_ja,
        'zh': cnt_zh,
        'ko': cnt_ko,
        'ar': cnt_ar,
        'th': cnt_th,
        'ru': cnt_ru,
    }

    lang, max_cnt = max(script_counts.items(), key=lambda kv: kv[1])
    if max_cnt >= SCRIPT_THRESHOLD:
        return lang
    return None

def make_snippet(lyrics: str, n_chars: int = 200) -> str:
    if not isinstance(lyrics, str) or not lyrics.strip():
        return ''
    return lyrics[:n_chars].replace('\n', ' ').strip()

# Ensure expected upstream schema
if 'length' not in df.columns:
    df['length'] = df['lyrics'].fillna('').astype(str).str.len()

snippets = df['lyrics'].map(make_snippet)
valid_mask = snippets.ne('')

# Pass 1: script-based shortcut
script_langs = snippets.map(detect_by_script)
script_mask = script_langs.notna() & valid_mask

langs = pd.Series('unknown', index=df.index, dtype='object')
langs.loc[script_mask] = script_langs.loc[script_mask]

print(f'Pass 1 (script check): {int(script_mask.sum())}/{len(df)} songs classified')

# Pass 2: FastText only for remaining rows
remaining_mask = valid_mask & ~script_mask
if remaining_mask.any():
    labels, _scores = ft_model.predict(snippets[remaining_mask].tolist(), k=1)
    langs.loc[remaining_mask] = [lbls[0].replace('__label__', '') for lbls in labels]
print(f'Pass 2 (FastText):     {int(remaining_mask.sum())}/{len(df)} songs classified')

# Build output
lang_df = df.copy()
lang_df['original_lang'] = langs

# Write with explicit column order to guarantee stable schema
out_cols = ['artist', 'title', 'spotify_uri', 'lyrics', 'length', 'original_lang']
lang_df = lang_df.loc[:, out_cols]
lang_df.to_csv(LANG_OUT, index=False)

print(f'\nSaved {len(lang_df)} rows -> {LANG_OUT}')
print('\nLanguage distribution:')
print(lang_df['original_lang'].value_counts().to_string())

print('\nScript check vs FastText breakdown:')
print(f"  script check : {int(script_mask.sum())}")
print(f"  fasttext     : {int(remaining_mask.sum())}")
print(f"  unknown      : {int((lang_df['original_lang'] == 'unknown').sum())}")

display(lang_df.head(10))

Loaded 694 songs from d:\Users\Documents\GitHub\lyrics_analysis\data\processed\00_lyrics.csv
Pass 1 (script check): 193/694 songs classified
Pass 2 (FastText):     459/694 songs classified

Saved 694 rows -> d:\Users\Documents\GitHub\lyrics_analysis\data\processed\01_lyrics_lang.csv

Language distribution:
original_lang
es         276
en         162
ja         156
unknown     42
ko          18
zh          16
tr           3
pt           3
it           2
sw           2
ru           2
gd           1
lmo          1
vi           1
tl           1
ar           1
eo           1
la           1
de           1
hr           1
fr           1
he           1
id           1

Script check vs FastText breakdown:
  script check : 193
  fasttext     : 459
  unknown      : 42


,artist,title,spotify_uri,lyrics,length,original_lang
0,"Ryan Castro, Kapo, Gangsta",LA VILLA,2ZyrAym0sRLwt4PhGotHuI,"Kapo, Ryan Castro, Gangsta\nQué chimba, SOG\nT...",2484,es
1,Bad Bunny,BAILE INoLVIDABLE,2lTm559tuIvatlT1u0JYG2,Pensaba que contigo iba a envejecer\nEn otra v...,2083,es
2,"El Bogueto, Yung Beef",Cuando No Era Cantante,6N2iccqxRInhTLHc2Fu3W0,"Jajajajaja\nJaja, QueHicisteBella\n\nComo ante...",2515,es
3,La T y La M,Soy Favela,3TfRpsYPQSXqqramSoWlNg,¡Ay!\n\nHe caminado tantas calles pero no he e...,1429,es
4,Max Carra,UWAIE - versión cumbia,6UO9DhCUq4ZbQz7PXwelFV,NaN,0,unknown
5,"Lauta, Amigo de Artistas, Tote",Puñaladas,4AL4EamHEBKPpdcFRkYdXN,"Si tu espejo hablara, te diría esto\n\nEsa car...",1350,es
6,Bad Bunny,DtMF,3sK8wGT43QFpWrvNQsrQya,"Eh, eh, eh, eh\n\nOtro sunset bonito que veo e...",2403,es
7,"Roze Oficial, Max Carra, Valen, RAMKY EN LOS C...",Tu jardín con enanitos,6X8DTIJEgHUjZynuds0E2f,"Con este tema, te voy a hacer viajar en el tie...",1576,es
8,Bad Bunny,VOY A LLeVARTE PA PR,59D4DOkspUbWyMmbAPQkxZ,"Acho, PR es otra cosa\nYo la conocí en Miami, ...",1923,es
9,"El Bogueto, Anuel AA, Fuerza Regida, Yung Beef",Cuando No Era Cantante - Remix,2pbSCYzxrG0wa6qcj8IyiE,Uh-uh-uh\nHow pretty that ass loo-oo-ooks\nIn ...,5006,en
